GENERATION OF TEXTUAL VERSION OF SLIDES

In [ ]:
from transformers import AutoTokenizer, AutoModel
from IPython.display import display
from IPython.display import Markdown
from PIL import Image,ImageDraw
from openai import OpenAI
from tqdm import tqdm
import pandas as pd
import textwrap
import base64
import torch
import json
import ast
import re
import io

c:\Users\user0\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

# Include your API key file path here 
with open('gpt_key.txt', 'r') as file:
    GPT_API_KEY = file.read()
client = OpenAI(api_key=GPT_API_KEY)

def extract_last_list(text):
    # Find all bracketed list-like patterns
    matches = re.findall(r'\[.*?\]', text, re.DOTALL)

    if matches:
        last_list_str = matches[-1]
        try:
            # Use literal_eval to safely convert it to a Python list
            return ast.literal_eval(last_list_str)
        except Exception as e:
            raise ValueError(f"Error parsing list: {e}")
            return None
    else:
        raise ValueError("No list found.")
        return None
  
def encode_image(image_path,width_size=800):
  img = Image.open(image_path)

  if img.width > width_size:
    ratio = width_size/img.width
    new_size = (width_size, int(img.height * ratio))
    img = img.resize(new_size)
  
  buffered = io.BytesIO()
  img.save(buffered, format="PNG", optimize=True)
  img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

  return img_b64

def gpt_big_call(model,user_content,n_responses=1,T=1,system_prompt="You are a helpful assistant."):
  flag_completion = True
  cost_consumption = 0
  while flag_completion:
    try:
      completion = client.chat.completions.create(
          model=model,
          messages=[
              {"role": "system", "content": system_prompt},
              {"role":"user","content":user_content}
          ],
          n=n_responses,
          temperature=T
      )
      gpt_response = completion.choices
      cost_consumption += completion.usage.prompt_tokens*0.00000125
      cost_consumption += completion.usage.completion_tokens*0.00001
      flag_completion = False 
    except Exception as e:
      print("Failed with e: {}".format(e))
  return gpt_response,cost_consumption

def extract_slide_types(main_string):
  """
  Extracts slide types (e.g., "TITLE_SLIDE", "ELABORATION_SLIDE")
  from a string using regex.

  Args:
    main_string: The string to search within.

  Returns:
    A list containing all identified slide type strings.
  """
  # The regex pattern looks for a quoted string within square brackets.
  # "(.+?)" captures any characters non-greedily inside the quotes.
  pattern = r'\["(.+?)"\]'
  matches = re.findall(pattern, main_string)
  return matches

In [19]:
gpt_model = "gpt-4.1"
total_cost = 0

In [20]:
# Load model from HuggingFace Hub
tokenizer_sim = AutoTokenizer.from_pretrained('BAAI/bge-large-en-v1.5')
model_sim = AutoModel.from_pretrained('BAAI/bge-large-en-v1.5')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

model_sim.to(device)
model_sim.eval()

device: cuda


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

ALL CONTENTS PROCESSING

In [ ]:
# This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments
# For a new course, you need to include here the information about the lecture materials included in the course (contentsid)
# the number of pages of each material (pages) and the name of the corresponding class (title)
course_id = 207
course_materials = pd.read_csv("./pid{}_material_info.csv".format(course_id))[["contentsid","pages","title"]]

for n_material in range(course_materials.shape[0]):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]
    print("ID: {}".format(material_id))
    print("N pages: {}".format(max_pages))
    print("Title: {}".format(title))

    title_prompt = """The title of a university lecture is: {}
TRANSLATE it into ENGLISH.
It should be a single element inside of a python list:
MAIN GOAL: ["main_learning_goal"]""".format(title)

    main_discourses = []

    response, money = gpt_big_call(gpt_model,title_prompt,10,1)
    total_cost += money

    for i in tqdm(range(10)):
        response_text = response[i].message.content
        try:
            learning_objectives = extract_last_list(response_text)
        except: 
            continue
        main_discourses.append(learning_objectives[0])

    encoded = tokenizer_sim(main_discourses, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model_sim(**encoded)
        embs = out[0][:,0]
        embs = torch.nn.functional.normalize(embs, p=2, dim=1)
    co_ref = embs@embs.T
    idx_result = co_ref.sum(0).argmax()
    print(main_discourses[idx_result])

    with open("./Agent_resources/Titles/pid{}_{}_title.txt".format(course_id,material_id), "w", encoding="utf-8") as f:
        f.write(main_discourses[idx_result])

    init_prompt = """You are writing a SMALL BOOK from your lecture slides.

The BOOK contain STRUCTURED information, using heading, subheadings and DETAILED explanations.

The BOOK DO NOT CONTAIN VISUAL MEDIA, so your EXPLANATION SHOULD REPLACE this information. DON'T RELY ON students looking for the corresponding media in the slides.


Current lecture is: {}.

The REPHRASED CONTENTS MUST INCLUDE: 
1. ALL the information specified in the slides. It can be elaborated or summarized ACCORDING to the MAIN PURPOSE of the slides.
2. Detailed descriptions and explanations of any data, equations, charts, or graphs (VISUAL MEDIA).

ELABORATE EXTENSIVELY THE CONTENTS to enhance readers' understanding (Section headers do not need elaboration).
ENSURE ALL YOUR WRITTEN INFORMATION IS IMPLICITLY OR EXPLICITLY STATED IN THE SLIDE (LIMITS OF THE ELABORATION OF THE CONTENTS).

Use Narrative Style (structured explanations). Use numbers for identifying slides (e.g., SLIDE 1).
VERIFY YOU ONLY USE ENGLISH.
VERIFY YOU DONT USE TEXTUAL REFERENCE TO INEXISTENT MEDIA (e.g., "the image shows a team working in a project", "a dog running in accompanying figure").
YOU MUST DESCRIBE THE CONTENTS INSTEAD, INCLUDING IMPLICIT INFORMATION ALIGNED TO THE LEARNING GOAL OF THE SLIDE (e.g., "Projects require team working which is a challenge", "Average dogs can run at a speed OF 40 km/h").
ENCLOSE ALL EQUATIONS, FORMULAS, and VARIABLES in double square brackets like this: [[ x + y = z ]].

OUTPUT FORMAT (NO NEED FOR INTRODUCTION OF THE CONTENT, e.g., "Here's the rephrased content from the slide, formatted as part of a book:"):

SLIDE 1: Subheading

Detailed explanation

SLIDE 2: Subheading

Detailed explanation

...
""".format(main_discourses[idx_result])


    user_content =[{"type":"text","text":init_prompt}]
    for i in range(1,max_pages+1):
        # This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments (data delivered upon request)
        # For a new course, you need to include here the lecture materials' slides or the educational documents' chunks in an image format.
        # For only-text cases you need to modify the code
        img = encode_image('./pid{}_materials/material_{}_page_{}.png'.format(course_id,material_id,i),1200)
        new_img = {"type": "image_url", "image_url":{"url": f"data:image/png;base64,{img}"}}
        user_content.append(new_img)


    # Four datasets for training and testing
    # In case of including more data for the training process, 
    # you can generate more sets (e.g., text_5, text_6, ...)    
    response, money = gpt_big_call(gpt_model,user_content,1,1,"You are a university professor.")
    total_cost += money

    with open("./Agent_resources/Text_versions/pid{}_{}_text_1.txt".format(course_id,material_id), "w", encoding="utf-8") as f:
        f.write(response[0].message.content)

    response, money = gpt_big_call(gpt_model,user_content,1,1,"You are a university professor.")
    total_cost += money

    with open("./Agent_resources/Text_versions/pid{}_{}_text_2.txt".format(course_id,material_id), "w", encoding="utf-8") as f:
        f.write(response[0].message.content)

    response, money = gpt_big_call(gpt_model,user_content,1,1,"You are a university professor.")
    total_cost += money

    with open("./Agent_resources/Text_versions/pid{}_{}_text_3.txt".format(course_id,material_id), "w", encoding="utf-8") as f:
        f.write(response[0].message.content)

    response, money = gpt_big_call(gpt_model,user_content,1,1,"You are a university professor.")
    total_cost += money

    with open("./Agent_resources/Text_versions/pid{}_{}_text_4.txt".format(course_id,material_id), "w", encoding="utf-8") as f:
        f.write(response[0].message.content)

    print("##############################################")
    print(f"total cost: {total_cost}")
    print("##############################################")

ID: 4eff0720836a198b6174eecf02cbfdbf
N pages: 28
Title: 第０回：授業の準備


100%|██████████| 10/10 [00:00<00:00, 9320.68it/s]


Session 0: Course Preparation
##############################################
total cost: 0.11004250000000002
##############################################
ID: 5a4be1fa34e62bb8a6ec6b91d2462f5a
N pages: 15
Title: 第７回：離散時間システム～線形時不変システム，差分方程式～


100%|██████████| 10/10 [00:00<?, ?it/s]


Session 7: Discrete-Time Systems – Linear Time-Invariant Systems and Difference Equations
##############################################
total cost: 0.20049500000000003
##############################################
ID: 69f62956429865909921fa916d61c1f8
N pages: 19
Title: 第５回：離散時間信号とZ変換


...

Properties of the Z-Transform and Inverse Z-Transform
##############################################
total cost: 1.1579475000000001
##############################################
ID: cceff8faa855336ad53b3325914caea2
N pages: 17
Title: 第４回：連続時間信号の標本化


100%|██████████| 10/10 [00:00<?, ?it/s]


Lecture 4: Sampling of Continuous-Time Signals
##############################################
total cost: 1.248755
##############################################
ID: fb642b781020b2aaeb1a7cee29acc915
N pages: 17
Title: 第８回：離散時間システム～周波数特性～


100%|██████████| 10/10 [00:00<?, ?it/s]


Lecture 8: Discrete-Time Systems – Frequency Characteristics –
##############################################
total cost: 1.34064125
##############################################


SLIDE SECTIONS

In [ ]:
type_prompt = f"""Extract the FUNCTIONAL TYPOLOGY of the provided SLIDES. The main learning goal of the lecture is: {main_discourses[0]}.

-------------------------------------------------------------------------------

The extracted TYPOLOGY MUST BE included in one of the following categories:

* TITLE_SLIDE: 
Characteristics: Prominent presentation title, presenter name, date, course information; often includes institutional branding or logos.
Intended purpose: Sets overall context, introduces the topic and presenter, establishes the lecture's identity and branding.
Examples of key characteristics: "Introduction to Quantum Physics", "Dr. John Smith - Fall 2023", "Course Name: [Code]" 

* AGENDA_SLIDE:
Characteristics: A concise, bulleted list of the main topics to be covered in the lesson (not the course); may include optional timings for each section or visual progress indicators (e.g., checkmarks, arrows, highlighting current topic).
Intended purpose: Provides a clear roadmap for the current lecture, and aids in the mental organization of incoming information.
Examples of key characteristics: "Lecture Outline:", "What We'll Cover Today:", "Key Topics:", "My case is based on three main points..." 

* HEADER_SLIDE:
Characteristics: Short textual phrases or a distinct graphic explicitly signaling a shift between major topics, sections, or logical arguments; often uses phrases like "Moving on to...", "Next, we'll discuss...", or "Having examined...".
Intended purpose: Guides the audience smoothly through the lecture flow, maintains coherence between disparate topics, and signals structural changes to aid comprehension.
Examples of key characteristics: "Moving On To...", "Next: Applications", "Section 2: Methodology", "Methodology", "Exercises", "Turning to the next point...", "Another important consideration is that..."

* ELABORATION_SLIDE:
Characteristics: Presents a concise definition of a term, concept, or theory. It may provide detailed descriptions of processes, mechanisms, theories, comparisons, or cause-and-effect relationships. It may include text, structured text and visuals in similar proportions.
Intended purpose: Introduces fundamental vocabulary, ensures shared understanding of core concepts. To deepen understanding of a concept, principle, or phenomenon. To clarify ambiguities and provide further detail beyond a basic definition. To illustrate the internal workings, relationships, or implications of a topic.
Examples of key characteristics: "Definition: Photosynthesis", "Steps of Photosynthesis", "What is Dual Channeling?", "Key Term: Pedagogy", "signal key concepts or define key terms", "define important concepts", "Steps of Photosynthesis", "Characteristics of good designs"

* IMAGE_SLIDE:
Characteristics: A prominent graphic, diagram, or image that visually explains or provides evidence for a textual statement (e.g., at the top of the slide).
Intended purpose: Emphasizes a core message, provides immediate visual support for a claim, and aids comprehension by integrating text and visuals effectively. Often used to present findings or arguments.
Examples of key characteristics: "Climate Change is Accelerating (Graph of Global Temps)", "Effective Slide Design (Image of a well-designed slide)", "assertion-visual evidence approach", "present an assertion... with a graphic that explains that assertion"

* TABLE_GRAPH_SLIDE:
Characteristics: Mainly displays quantitative or qualitative information through tables, charts (e.g., bar, pie, line), or raw data (no bullet points).
Intended purpose: Presents very structured information, supports evidence-based discussion, or interpretation. Aids comprehension by structuring and summarizing large amounts of information into tables, graphs.
Examples of key characteristics: "Sales Figures Q3 2023 (Bar Chart)", "Survey Results (Pie Chart)", "A graph illustrating selected information", "Analysis of results", "Table summarizing explained methods"

* EXAMPLE_SLIDE:
Characteristics: Visual or textual information aimed to exemplify a concept or process. It may use step-by-step visuals such as flowcharts, numbered diagrams, or animated sequences to depict a process, procedure, or sequence of events. 
Intended purpose: Clarifies abstract concepts, provides concrete instances, explains sequences, procedures, or complex operations, supporting procedural understanding, application, and analysis.
Examples of key characteristics: "Graph showing population growth","Steps in DNA Replication (animated)", "Solving Quadratic Equations (step-by-step)", "showing processes, or building models in real time", "flowchart to visually illustrate the steps" 

* QUESTION_SLIDE:
Characteristics: Contains open-ended questions, exercises, challenges, prompts for discussion, brainstorming activities, or polls. It does not contain the solution.
Intended purpose: Promotes active thinking, checks immediate understanding, facilitates application of theoretical knowledge to real-world contexts, develops critical thinking, problem-solving, and decision-making skills.
Examples of key characteristics: "What are your thoughts?", "Solve this problem:","Brainstorm solutions for X", "Poll: A, B, or C?", "Presenting slide content as questions or challenges", "Case Study: Patient X", "Decision Point: Choose A or B", "Interactive Scenario"

* CONCLUSION_SLIDE:
Characteristics: Key takeaways, main points, or core arguments summarized concisely, typically in bullet points or a brief paragraph.
Intended purpose: Consolidates learning, reinforces the most critical messages, and provides a sense of closure for the lecture.
Examples of key characteristics: "Key Takeaways", "Conclusions", "In Summary", "Recap", "I'd like to finish by...", "Summarise the key points covered" 

* Q&A_SLIDE:
Characteristics: A clear prompt for questions, often accompanied by presenter's contact information, a visual cue for audience interaction, or a statement inviting comments.
Intended purpose: Facilitates audience interaction, encourages clarification of concepts, and provides explicit avenues for follow-up engagement.
Examples of key characteristics: "Questions?", "Discussion Points", "Contact Me", "invite them to comment or ask questions"

* NEXT_CLASS_SLIDE:
Characteristics: Explicit instructions for follow-up activities, assignments, required readings, additional resources, or prompts for independent study.
Intended purpose: Directs future learning, reinforces the application of lecture content, and provides resources for continued engagement beyond the lecture.
Examples of key characteristics: "Further Reading", "Preparation for next class", "Assignment Due", "Explore More", "Note things to follow up for future reference" 

-------------------------------------------------------------------------------

OUTPUT FORMAT: (Only one type per slide)

Slide 1: ["Slide_type"]
Slide 2: ["Slide_type"]
Slide 3: ["Slide_type"]
Slide 4: ["Slide_type"]
Slide 5: ["Slide_type"]
...
"""

user_content =[{"type":"text","text":type_prompt}]
for i in range(1,max_pages+1):
    # This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments (data delivered upon request)
    # For a new course, you need to include here the lecture materials' slides or the educational documents' chunks in an image format.
    # For only-text cases you need to modify the code
    img = encode_image('./pid{}_materials/material_{}_page_{}.png'.format(course_id,material_id,i),1200)
    new_img = {"type": "image_url", "image_url":{"url": f"data:image/png;base64,{img}"}}
    user_content.append(new_img)